# LSTM/GRU Core Module
A Notebook for recurrent sequential processing

## 1. Imports

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import json

print('True')

True


In [6]:
import os
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image

# paths
CSV_PATH    = '/kaggle/input/datasets/manjushwarkhairkar/cleaned-fashion-tech-myntra-dataset-with-features/fashion_preprocessed.csv'
VOCAB_DIR   = '/kaggle/input/models/manjushwarkhairkar/vocabs-and-category-configes/other/default/1/vocab_and_categories'
SPLITS_PATH = VOCAB_DIR

# reload df with indices
df = pd.read_csv(os.path.join(VOCAB_DIR, 'fashion_with_indices.csv'))
df = df.dropna(subset=['proc_img_path', 'proc_sketch_path']).reset_index(drop=True)
print(f"Loaded {len(df)} rows")
print()
print(f"Columns: {df.columns.tolist()}")

# reload vocabs
with open(os.path.join(VOCAB_DIR, 'colour_vocab.json'))   as f:
    colour_vocab   = json.load(f)
with open(os.path.join(VOCAB_DIR, 'category_vocab.json')) as f:
    category_vocab = json.load(f)
print()

print(f"Colour vocab size  : {len(colour_vocab)}")
print(f"Category vocab size: {len(category_vocab)}")

Loaded 14311 rows

Columns: ['p_id', 'name', 'price', 'colour', 'brand', 'img', 'ratingCount', 'avg_rating', 'description', 'p_attributes', 'tok_len', 'attr_keys', 'img_path', 'proc_img_path', 'proc_sketch_path', 'category', 'colour_idx', 'category_idx']

Colour vocab size  : 52
Category vocab size: 27


## 3. Rebuilding dataset + dataloader

In [9]:
# Cell 3 — Rebuild dataset + dataloader (recreate splits, don't load)
from torch.utils.data import random_split

img_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])
sketch_transform = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

class MyntraSketchDataset(Dataset):
    def __init__(self, dataframe, img_tfm=None, sketch_tfm=None):
        self.df         = dataframe.reset_index(drop=True)
        self.img_tfm    = img_tfm
        self.sketch_tfm = sketch_tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(row['proc_img_path']).convert('RGB')
        sketch = Image.open(row['proc_sketch_path']).convert('RGB')
        if self.img_tfm:    img    = self.img_tfm(img)
        if self.sketch_tfm: sketch = self.sketch_tfm(sketch)
        return {
            'sketch'      : sketch,
            'image'       : img,
            'name'        : row['name'],
            'colour'      : row['colour'],
            'colour_idx'  : torch.tensor(int(row['colour_idx']),   dtype=torch.long),
            'category_idx': torch.tensor(int(row['category_idx']), dtype=torch.long),
            'idx'         : idx
        }

full_dataset = MyntraSketchDataset(df, img_tfm=img_transform, sketch_tfm=sketch_transform)

# recreate splits with same seed — guarantees identical 80/10/10 every time
total   = len(full_dataset)
n_train = int(0.8 * total)
n_val   = int(0.1 * total)
n_test  = total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

PIN          = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=PIN)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=PIN)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Train: 11448 | Val: 1431 | Test: 1432
Train batches: 358
Val   batches: 45


## 4. Encoder + Embedder

In [11]:
# Cell 4 — Rebuild encoder + embedder fresh (no saved weights needed)
import torchvision.models as models

class SketchEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone      = nn.Sequential(*list(resnet.children())[:-2])
        self.input_adapter = nn.Conv2d(1, 3, kernel_size=1, bias=False)
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, sketch):
        x = self.input_adapter(sketch)
        x = self.backbone(x)
        return x                          # [B, 512, 7, 7]

class StyleEmbedding(nn.Module):
    def __init__(self, colour_vocab_size, category_vocab_size, embed_dim=64):
        super().__init__()
        self.colour_emb   = nn.Embedding(colour_vocab_size,   embed_dim, padding_idx=0)
        self.category_emb = nn.Embedding(category_vocab_size, embed_dim, padding_idx=0)
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

    def forward(self, colour_idx, category_idx):
        c     = self.colour_emb(colour_idx)
        s     = self.category_emb(category_idx)
        fused = torch.cat([c, s], dim=-1)
        return self.fusion(fused)          # [B, 128]

encoder        = SketchEncoder()
style_embedder = StyleEmbedding(
    colour_vocab_size   = len(colour_vocab),
    category_vocab_size = len(category_vocab),
    embed_dim           = 64
)

encoder.eval()
style_embedder.eval()
print("Encoder and StyleEmbedder initialized fresh!")

Encoder and StyleEmbedder initialized fresh!


In [13]:
# Cell 5 — LSTMCore class
class LSTMCore(nn.Module):

    def __init__(self,
                 encoder_dim  = 512,   # from ResNet-18 output
                 style_dim    = 128,   # from StyleEmbedding output
                 hidden_size  = 256,   # GRU hidden state size
                 num_layers   = 2,     # stacked GRU layers
                 dropout      = 0.1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # project encoder spatial features → 1D context vector
        # [B, 512, 7, 7] → global avg pool → [B, 512] → linear → [B, 256]
        self.encoder_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),      # [B, 512, 1, 1]
            nn.Flatten(),                 # [B, 512]
            nn.Linear(encoder_dim, hidden_size),
            nn.ReLU()
        )

        # project style vector [B, 128] → [B, 256]
        self.style_proj = nn.Sequential(
            nn.Linear(style_dim, hidden_size),
            nn.ReLU()
        )

        # fuse encoder + style → single input to GRU
        # [B, 256] + [B, 256] → [B, 512] → [B, 256]
        self.input_proj = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # stacked GRU — the memory core
        self.gru = nn.GRU(
            input_size  = hidden_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )

        # layer norm for stable training
        self.layer_norm = nn.LayerNorm(hidden_size)

    def forward(self, encoder_features, style_vector, hidden=None):
        # encoder_features : [B, 512, 7, 7]
        # style_vector     : [B, 128]
        # hidden           : [num_layers, B, hidden_size] or None

        # project encoder features → [B, 256]
        enc_ctx = self.encoder_proj(encoder_features)

        # project style vector → [B, 256]
        sty_ctx = self.style_proj(style_vector)

        # fuse → [B, 256]
        fused = self.input_proj(torch.cat([enc_ctx, sty_ctx], dim=-1))

        # GRU expects [B, seq_len, input_size] — seq_len=1 per step
        fused = fused.unsqueeze(1)               # [B, 1, 256]

        # pass through stacked GRU
        out, hidden = self.gru(fused, hidden)    # out: [B, 1, 256]

        # squeeze seq dim + layer norm
        out = self.layer_norm(out.squeeze(1))    # [B, 256]

        return out, hidden
        # out    → [B, 256]  feeds into attention module
        # hidden → [num_layers, B, 256]  carried to next recurrent step

print('Done!')

Done!


In [15]:
# Cell 6 — Instantiate and inspect
lstm_core = LSTMCore(
    encoder_dim = 512,
    style_dim   = 128,
    hidden_size = 256,
    num_layers  = 2,
    dropout     = 0.1
)

print(lstm_core)

total     = sum(p.numel() for p in lstm_core.parameters())
trainable = sum(p.numel() for p in lstm_core.parameters() if p.requires_grad)
print(f"\nTotal params    : {total:,}")
print(f"Trainable params: {trainable:,}   ← all trained from scratch")

LSTMCore(
  (encoder_proj): Sequential(
    (0): AdaptiveAvgPool2d(output_size=1)
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
  )
  (style_proj): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
  )
  (input_proj): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (gru): GRU(256, 256, num_layers=2, batch_first=True, dropout=0.1)
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
)

Total params    : 1,085,696
Trainable params: 1,085,696   ← all trained from scratch


In [16]:
# Cell 7 — Sanity check with real batch
batch = next(iter(train_loader))

sketches      = batch['sketch']           # [32, 1, 224, 224]
colour_idx    = batch['colour_idx']       # [32]
category_idx  = batch['category_idx']    # [32]

with torch.no_grad():
    # step 1 — encoder
    features     = encoder(sketches)                        # [32, 512, 7, 7]
    # step 2 — style embedding
    style_vec    = style_embedder(colour_idx, category_idx) # [32, 128]
    # step 3 — lstm core
    out, hidden  = lstm_core(features, style_vec)           # [32, 256]

print(f"Encoder output shape : {features.shape}")
print(f"Style vector shape   : {style_vec.shape}")
print(f"LSTM output shape    : {out.shape}")
print(f"LSTM hidden shape    : {hidden.shape}")
print(f"\nExpected output : torch.Size([32, 256])")
print(f"Expected hidden : torch.Size([2, 32, 256])")
print(f"\nOutput match    : {out.shape    == torch.Size([32, 256])}")
print(f"Hidden match    : {hidden.shape == torch.Size([2, 32, 256])}")

Encoder output shape : torch.Size([32, 512, 7, 7])
Style vector shape   : torch.Size([32, 128])
LSTM output shape    : torch.Size([32, 256])
LSTM hidden shape    : torch.Size([2, 32, 256])

Expected output : torch.Size([32, 256])
Expected hidden : torch.Size([2, 32, 256])

Output match    : True
Hidden match    : True
